In [4]:
# 1. Load environment variables
from dotenv import load_dotenv
import os

load_dotenv()  # Load from .env file

# 2. Access keys
bearer_token = os.getenv("BEARER_TOKEN")
consumer_key = os.getenv("CONSUMER_KEY")
consumer_secret = os.getenv("CONSUMER_SECRET")
access_token = os.getenv("ACCESS_TOKEN")
access_token_secret = os.getenv("ACCESS_TOKEN_SECRET")

In [5]:
import nltk
nltk.download('punkt')
nltk.download('stopwords')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ishwa\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ishwa\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [6]:
# Install required packages
!pip install tweepy vaderSentiment

import tweepy
import pandas as pd
import time
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# API credentials
 

client = tweepy.Client(
    bearer_token=bearer_token,
    consumer_key=consumer_key, 
    consumer_secret=consumer_secret,
    access_token=access_token, 
    access_token_secret=access_token_secret
)

query = "(Apple OR Microsoft) -is:retweet"
tweets = []
limit = 10  # Reduced from 100 to avoid rate limits

try:
    # Adding error handling and rate limit awareness
    response = client.search_recent_tweets(
        query=query,
        max_results=limit,
        tweet_fields=['created_at', 'text']
    )
    
    if response.data:
        for tweet in response.data:
            tweets.append([tweet.created_at, tweet.text])
    
    # Create DataFrame
    df = pd.DataFrame(tweets, columns=["date", "text"])
    
    # If we have tweets, perform sentiment analysis
    if not df.empty:
        # Initialize sentiment analyzer
        analyzer = SentimentIntensityAnalyzer()
        
        # Calculate sentiment scores
        df['sentiment'] = df['text'].apply(lambda x: analyzer.polarity_scores(x)['compound'])
        
        # Classify sentiment
        df['label'] = df['sentiment'].apply(lambda x: 'positive' if x > 0.05 else ('negative' if x < -0.05 else 'neutral'))
        
        # Display results
        print(df[['date', 'text', 'sentiment', 'label']].head())
    else:
        print("No tweets were retrieved. Creating a sample DataFrame for demonstration.")
        # Create a sample DataFrame if no tweets were retrieved
        sample_data = {
            'date': ['2023-01-01', '2023-01-02', '2023-01-03'],
            'text': [
                'I love the new Apple products!', 
                'Microsoft services are down again, very frustrating.', 
                'The new tech releases are okay I guess.'
            ]
        }
        df = pd.DataFrame(sample_data)
        
        # Initialize sentiment analyzer
        analyzer = SentimentIntensityAnalyzer()
        
        # Calculate sentiment scores
        df['sentiment'] = df['text'].apply(lambda x: analyzer.polarity_scores(x)['compound'])
        
        # Classify sentiment
        df['label'] = df['sentiment'].apply(lambda x: 'positive' if x > 0.05 else ('negative' if x < -0.05 else 'neutral'))
        
        # Display results
        print("Sample data sentiment analysis:")
        print(df[['date', 'text', 'sentiment', 'label']])
    
except tweepy.TooManyRequests:
    print("Rate limit exceeded. Using sample data instead.")
    # Create a sample DataFrame if rate limited
    sample_data = {
        'date': ['2023-01-01', '2023-01-02', '2023-01-03'],
        'text': [
            'I love the new Apple products!', 
            'Microsoft services are down again, very frustrating.', 
            'The new tech releases are okay I guess.'
        ]
    }
    df = pd.DataFrame(sample_data)
    
    # Initialize sentiment analyzer
    analyzer = SentimentIntensityAnalyzer()
    
    # Calculate sentiment scores
    df['sentiment'] = df['text'].apply(lambda x: analyzer.polarity_scores(x)['compound'])
    
    # Classify sentiment
    df['label'] = df['sentiment'].apply(lambda x: 'positive' if x > 0.05 else ('negative' if x < -0.05 else 'neutral'))
    
    # Display results
    print("Sample data sentiment analysis:")
    print(df[['date', 'text', 'sentiment', 'label']])
    
except Exception as e:
    print(f"An error occurred: {e}")

Rate limit exceeded. Using sample data instead.
Sample data sentiment analysis:
         date                                               text  sentiment  \
0  2023-01-01                     I love the new Apple products!     0.6696   
1  2023-01-02  Microsoft services are down again, very frustr...    -0.4927   
2  2023-01-03            The new tech releases are okay I guess.     0.2263   

      label  
0  positive  
1  negative  
2  positive  


In [7]:
import re

# Add company label from text
def extract_company(text):
    text = text.lower()
    if 'apple' in text:
        return 'Apple'
    elif 'microsoft' in text:
        return 'Microsoft'
    else:
        return 'Unknown'

df['company'] = df['text'].apply(extract_company)

# Convert datetime to just date
df['date'] = pd.to_datetime(df['date']).dt.date

# Group by company and date, then take mean sentiment
daily_sentiment = df.groupby(['company', 'date'])['sentiment'].mean().reset_index()
daily_sentiment.rename(columns={'sentiment': 'avg_sentiment'}, inplace=True)

print(daily_sentiment.head())


     company        date  avg_sentiment
0      Apple  2023-01-01         0.6696
1  Microsoft  2023-01-02        -0.4927
2    Unknown  2023-01-03         0.2263
